<a href="https://colab.research.google.com/github/nyluje/TestTechnique/blob/main/Test_Technique_JCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/nyluje/TestTechnique/main/rapport.json -O rapport.json

--2026-04-23 17:37:40--  https://raw.githubusercontent.com/nyluje/TestTechnique/main/rapport.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 253492 (248K) [text/plain]
Saving to: ‘rapport.json’

rapport.json        100%[===================>] 247.55K  --.-KB/s    in 0.02s   

2026-04-23 17:37:40 (9.73 MB/s) - ‘rapport.json’ saved [253492/253492]



In [9]:
import pandas as pd

# Load the JSON file into a DataFrame
df = pd.read_json('rapport.json')

# Normalize the nested 'service_status' field into separate columns
df = pd.json_normalize(df.to_dict('records'))

# Display the DataFrame
df.head()

,timestamp,cpu_usage,memory_usage,latency_ms,disk_usage,network_in_kbps,network_out_kbps,io_wait,thread_count,active_connections,error_rate,uptime_seconds,temperature_celsius,power_consumption_watts,service_status.database,service_status.api_gateway,service_status.cache
0,2023-10-01 12:00:00+00:00,93,86,334,89,2541,2137,12,143,126,0.12,360000,84,356,online,degraded,online
1,2023-10-01 12:30:00+00:00,57,66,139,61,1171,1193,3,145,51,0.02,361800,58,253,online,online,online
2,2023-10-01 13:00:00+00:00,56,68,136,62,1316,1147,3,147,49,0.02,363600,59,242,online,online,online
3,2023-10-01 13:30:00+00:00,53,62,142,58,1090,1121,3,154,51,0.02,365400,62,240,online,online,online
4,2023-10-01 14:00:00+00:00,52,67,123,61,1086,1123,3,145,48,0.02,367200,59,249,online,online,online


In [2]:
!pip install langchain langchain-core langchain-community pandas scikit-learn pyod numpy





In [3]:
import pandas as pd

# Load the JSON file
df = pd.read_json('rapport.json')
df = pd.json_normalize(df.to_dict('records'))

# Convert timestamp to datetime (if needed)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Select numerical columns for anomaly detection
numerical_cols = [
    'cpu_usage', 'memory_usage', 'latency_ms', 'disk_usage',
    'network_in_kbps', 'network_out_kbps', 'io_wait',
    'thread_count', 'active_connections', 'error_rate',
    'uptime_seconds', 'temperature_celsius', 'power_consumption_watts'
]

In [13]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Extract numerical features
X = df[numerical_cols]

# Train Isolation Forest
clf = IsolationForest(contamination=0.05, random_state=42)  # Adjust contamination based on expected anomaly rate
clf.fit(X)

# Predict anomalies (-1 = anomaly, 1 = normal)
df['anomaly'] = clf.predict(X)
df['anomaly_score'] = clf.decision_function(X)  # Lower scores = more anomalous

In [14]:
from sklearn.ensemble import IsolationForest
import numpy as np

# Extract numerical features
X = df[numerical_cols]

# Train Isolation Forest
clf = IsolationForest(contamination=0.05, random_state=42)  # Adjust contamination based on expected anomaly rate
clf.fit(X)

# Predict anomalies (-1 = anomaly, 1 = normal)
df['anomaly'] = clf.predict(X)
df['anomaly_score'] = clf.decision_function(X)  # Lower scores = more anomalous

In [15]:
from langchain_core.tools import tool
import numpy as np
from sklearn.ensemble import IsolationForest
import pandas as pd

@tool
def explain_anomalies(df: pd.DataFrame, top_n: int = 5) -> str:
    """Explain the top N anomalies in the DataFrame."""
    anomalies = df[df['anomaly'] == -1].sort_values(by='anomaly_score').head(top_n)
    if anomalies.empty:
        return "No anomalies detected."

    explanation = []
    for _, row in anomalies.iterrows():
        explanation.append(
            f"Anomaly at {row['timestamp']}: "
            f"CPU={row['cpu_usage']}%, Memory={row['memory_usage']}%, "
            f"Latency={row['latency_ms']}ms, Temp={row['temperature_celsius']}°C. "
        )
    return "\n".join(explanation)

# Redefine numerical_cols here to ensure it's always present before IsolationForest calculation
numerical_cols = [
    'cpu_usage', 'memory_usage', 'latency_ms', 'disk_usage',
    'network_in_kbps', 'network_out_kbps', 'io_wait',
    'thread_count', 'active_connections', 'error_rate',
    'uptime_seconds', 'temperature_celsius', 'power_consumption_watts'
]

# Re-run Isolation Forest to ensure 'anomaly' and 'anomaly_score' columns are present
X_current = df[numerical_cols]
clf_current = IsolationForest(contamination=0.05, random_state=42)
clf_current.fit(X_current)
df['anomaly'] = clf_current.predict(X_current)
df['anomaly_score'] = clf_current.decision_function(X_current)

# Example usage for direct function call (for testing, etc.)
# If you intend for the agent to use this, you don't need this direct call.
explanation = explain_anomalies.func(df, top_n=3)
print(explanation)

Anomaly at 2023-10-03 14:00:00+00:00: CPU=91%, Memory=84%, Latency=384ms, Temp=89°C. 
Anomaly at 2023-10-02 13:00:00+00:00: CPU=99%, Memory=92%, Latency=327ms, Temp=87°C. 
Anomaly at 2023-10-11 04:30:00+00:00: CPU=67%, Memory=71%, Latency=199ms, Temp=62°C. 


In [30]:
import langchain as lchain
lchain.__version__

'1.2.15'